# Counts and metadata

Read a GEO accession as a counts matrix with per-cell annotations and study
metadata joined into an AnnData object for scanpy.

Reading counts needs the `counts` extra:

```bash
uv add 'seqout[counts]'
```

`.rds` files additionally need R on the PATH, with whichever package wrote the
object. That is only a fallback; most `.rds` files are read without it.

In [1]:
import pandas as pd

from seqout import connect, SeqoutListCounts

ACCESSION = "GSE297547"  # PBMCs from myasthenia gravis patients and controls

## Fetching counts from GEO

`SeqoutListCounts` resolves and groups files using API metadata. Downloads
start when a unit is read.

A unit is a group of files that read as one matrix: a 10x triplet
(`matrix.mtx` + `barcodes.tsv` + `features.tsv`), a CellRanger `.h5`, an
`.h5ad`, an `.rds`, or a delimited table.

In [2]:
counts = SeqoutListCounts(gse=ACCESSION)
manifest = counts.manifest()

print(f"{len(manifest)} units")
manifest[["unit", "sample", "format", "preferred", "has_metadata"]].head(8)

39 units


,unit,sample,format,preferred,has_metadata
0,GSM8994520:adt_C001,GSM8994520,h5ad,False,True
1,GSM8994520:rna_C001,GSM8994520,h5ad,True,True
2,GSM8994521:adt_C002,GSM8994521,h5ad,False,True
3,GSM8994521:rna_C002,GSM8994521,h5ad,True,True
4,GSM8994522:adt_C003,GSM8994522,h5ad,False,True
5,GSM8994522:rna_C003,GSM8994522,h5ad,True,True
6,GSM8994523:adt_C004,GSM8994523,h5ad,False,True
7,GSM8994523:rna_C004,GSM8994523,h5ad,True,True


`preferred` marks the default unit per sample. Alternate formats remain
selectable by unit label.

`has_metadata` reports a named annotation sidecar or embedded metadata.
`.h5ad` and `.rds` units carry embedded metadata.

In [3]:
manifest["format"].value_counts()

format
h5ad    38
tar      1
Name: count, dtype: int64

## Reading one sample

`matrix()` downloads the selected unit's files, caches them under
`~/.cache/seqout/counts/`, and reads them.

In [4]:
m = counts.matrix(sample=counts.units()[0].label)
m

CountMatrix(GSM8994520, single_cell, h5ad, 11032 cells x 25259 genes)

`CountMatrix` uses AnnData orientation: observations by genes. Observations
are cells for single-cell data and biological samples for bulk data.

In [5]:
m.summary

accession                           GSM8994520
kind                               single_cell
format                                    h5ad
observations                             11032
features                                 25259
sparse                                    True
obs_columns                                  8
metadata_fields    sample_id, tissue, sex, age
evidence                             h5ad file
source                GSM8994520_rna_C001.h5ad
Name: GSM8994520, dtype: object

`kind` is decided on evidence. Barcode-looking row
labels mean single-cell; row labels that are GSM accessions, or an observation
count no larger than the number of samples in the series, mean bulk. A dense
table with a few hundred columns could be a Smart-seq plate or a large bulk
study, and in that band `kind` reports `unknown`. Check
`evidence` when the answer matters.

## Cell-level metadata

`obs` carries whatever per-cell annotation the submitter deposited. For an
`.h5ad` or `.rds` that comes from inside the container; for a 10x unit it comes
from a sidecar file, joined on the cell label.

In [6]:
m.obs.head()

,type1,type2,type3,title,tissue,age,Sex,sample
C001_13,Control,WTA,C001,PBMC from C01 sample,PBMC,45,male,GSM8994520
C001_64,Control,WTA,C001,PBMC from C01 sample,PBMC,45,male,GSM8994520
C001_84,Control,WTA,C001,PBMC from C01 sample,PBMC,45,male,GSM8994520
C001_398,Control,WTA,C001,PBMC from C01 sample,PBMC,45,male,GSM8994520
C001_800,Control,WTA,C001,PBMC from C01 sample,PBMC,45,male,GSM8994520


`metadata_fields` maps annotation names to a fixed vocabulary and records
the source columns. This distinguishes labels derived from `disease_status`
from those derived from `treatment`.

In [7]:
m.metadata_fields

{'sample_id': ['sample'], 'tissue': ['tissue'], 'sex': ['Sex'], 'age': ['age']}

The submitter's labels `type1`, `type2`, and `type3` have no vocabulary match,
so `metadata_fields` is empty. The source columns remain in `obs`.

Recognisable names populate the mapping. `seurat_clusters`, `leiden`, and
`louvain` map to clusters; cell-type annotation requires cell-type labels.

In [8]:
m.obs["type1"].value_counts().head()

type1
Control    11032
Name: count, dtype: int64

## Picking an assay

This CITE-seq study has gene-expression and antibody-panel `.h5ad` files.
`SeqoutListCounts` defaults to `assay="rna"` so alphabetic file order cannot
select the antibody panel. Select the panel explicitly to read it.

In [9]:
adt = SeqoutListCounts(gse=ACCESSION, assay="adt")
adt_m = adt.matrix(sample=adt.units()[0].label)

pd.DataFrame({"rna": m.summary, "adt": adt_m.summary})

,rna,adt
accession,GSM8994520,GSM8994520
kind,single_cell,single_cell
format,h5ad,h5ad
observations,11032,11032
features,25259,9
sparse,True,True
obs_columns,8,56
metadata_fields,"sample_id, tissue, sex, age","sample_id, celltype, tissue, sex, age"
evidence,h5ad file,h5ad file
source,GSM8994520_rna_C001.h5ad,GSM8994520_adt_C001.h5ad


In [10]:
adt_m.obs["Cell_Type_Experimental"].value_counts().head()

Cell_Type_Experimental
B                 5207
T_CD4_memory      4860
T_CD8_memory       890
Natural_killer      70
T_gamma_delta        4
Name: count, dtype: int64

In [11]:
adt.close()

## Study metadata from seqout.org

The API supplies donor characteristics, study conditions, and linked publications.

In [12]:
sq = connect("api")
project = sq.fetch_project_metadata(ACCESSION)

print(project.title)
print()
print("organisms :", project.organisms)
print("published :", project.published_at)
print("modality  :", project.single_cell_modality)
print("samples   :", len(project.samples_ref))

Single-cell RNA sequencing of PBMCs from patients with myasthenia gravis and healthy controls

organisms : ['Homo sapiens']
published : 2025-06-30
modality  : scRNA-seq
samples   : 25


`counts.design` returns sample characteristics indexed by accession. GEO
uses submitter-defined key/value pairs, so columns vary by study.
`seqout.sample_frame(...)` builds the same frame from a sample list.

In [13]:
counts.design.head()

,title,tissue,age,Sex
sample,,,,
GSM8994520,PBMC from C01 sample,PBMC,45,male
GSM8994521,PBMC from C02 sample,PBMC,55,male
GSM8994522,PBMC from C03 sample,PBMC,63,female
GSM8994523,PBMC from C04 sample,PBMC,57,female
GSM8994524,PBMC from C05 sample,PBMC,70,female


## Fetching counts for all samples

`matrix()` joins sample characteristics onto `obs` by GSM accession.
Per-cell annotation wins when both sources define a column because it is
more specific.

In [14]:
m.obs[["type1", "sample", "tissue", "age", "Sex"]].head()

,type1,sample,tissue,age,Sex
C001_13,Control,GSM8994520,PBMC,45,male
C001_64,Control,GSM8994520,PBMC,45,male
C001_84,Control,GSM8994520,PBMC,45,male
C001_398,Control,GSM8994520,PBMC,45,male
C001_800,Control,GSM8994520,PBMC,45,male


In [15]:
adata = m.to_anndata()
adata

AnnData object with n_obs × n_vars = 11032 × 25259
    obs: 'type1', 'type2', 'type3', 'title', 'tissue', 'age', 'Sex', 'sample'
    var: 'Raw_Reads-0', 'Raw_Molecules-0', 'Raw_Seq_Depth-0', 'RSEC_Adjusted_Molecules-0', 'RSEC_Adjusted_Reads_non-singleton-0', 'RSEC_Adjusted_Molecules_non-singleton-0', 'Raw_Reads-1', 'Raw_Molecules-1', 'Raw_Seq_Depth-1', 'RSEC_Adjusted_Molecules-1', 'RSEC_Adjusted_Reads_non-singleton-1', 'RSEC_Adjusted_Molecules_non-singleton-1', 'Raw_Reads-10', 'Raw_Molecules-10', 'Raw_Seq_Depth-10', 'RSEC_Adjusted_Molecules-10', 'RSEC_Adjusted_Reads_non-singleton-10', 'RSEC_Adjusted_Molecules_non-singleton-10', 'Raw_Reads-11', 'Raw_Molecules-11', 'Raw_Seq_Depth-11', 'RSEC_Adjusted_Molecules-11', 'RSEC_Adjusted_Reads_non-singleton-11', 'RSEC_Adjusted_Molecules_non-singleton-11', 'Raw_Reads-12', 'Raw_Molecules-12', 'Raw_Seq_Depth-12', 'RSEC_Adjusted_Molecules-12', 'RSEC_Adjusted_Reads_non-singleton-12', 'RSEC_Adjusted_Molecules_non-singleton-12', 'Raw_Reads-13', 'Raw_Mol

From here it is an ordinary AnnData: `sc.pp.filter_cells`, `sc.pp.normalize_total`
and the rest of the scanpy pipeline all work, and the donor covariates you just
attached are available for grouping and for regressing out batch.

The provenance of the matrix is recorded in `uns` so a downstream reader can
tell where the numbers came from.

In [16]:
adata.uns["seqout"]

{'accession': 'GSM8994520',
 'kind': 'single_cell',
 'format': 'h5ad',
 'source': 'GSM8994520_rna_C001.h5ad',
 'evidence': ['h5ad file'],
 'has_metadata': True,
 'metadata_fields': {'sample_id': ['sample'],
  'tissue': ['tissue'],
  'sex': ['Sex'],
  'age': ['age']}}

## Whole studies, and other formats

`matrices()` reads every preferred unit and returns a dict keyed by unit label.
It prefetches files in one threaded call.

```python
mats = counts.matrices()               # every sample
adata = counts.anndata()               # concatenated, inner join on genes
paths = counts.raw(sample="GSM...")    # just the files, no parsing
```

`matrices()` skips a unit it cannot read, so compare the number of results
against the manifest.

## Choosing which samples to read

A large series mixes tissues, conditions and assays, and you rarely want all of
it. `samples()` puts the study's samples through the harmonised cohort search
and keeps the ones that ship a readable unit. It answers with the `unit` and
the `format` that `matrix()` would read, most cells first.

```python
liver = counts.samples(tissue="liver", min_cell_count=1000)
liver[["sample", "unit", "format", "cells", "tissue"]]

m = counts.matrix(sample=liver["unit"].iloc[0])
```

Cohort filters read harmonised data. Unharmonised samples are excluded, so
an empty result can reflect missing harmonisation.

## Binding samples together

`bind_counts` joins matrices on the genes they share and returns one AnnData,
cells by genes. `max_cells` caps the cells kept per matrix, drawn at random;
give `seed` for a repeatable draw.

```python
from seqout import bind_counts

merged = bind_counts(counts.matrices(), max_cells=1200, seed=0)
merged.obs["sample"].value_counts()
```

Each cell name gets its unit label as a suffix, so two samples that use the
same barcodes stay apart. `counts.anndata()` is the same call over every
preferred unit.

The R client uses genes by cells for Seurat. The Python client uses cells by genes.

## Labelling clusters by marker set

`quick_annotation` scores each cell as the mean expression of a marker set's
genes, averages that within each cluster, and gives the cluster the name of its
best set.

```python
from seqout import quick_annotation

markers = {"neuron": ["NEUROD2", "TBR1"], "microglia": ["CX3CR1", "C1QA"]}
labels = quick_annotation(adata, adata.obs["leiden"], markers)

adata.obs["celltype"] = labels[adata.obs["leiden"].astype(str)].to_numpy()
```

The scores are not scaled across the sets, so a set of housekeeping genes can
beat a small specific one. Read `labels.attrs["scores"]` before you trust a
label. Specific markers work best: genes that one cell type expresses and the
others do not.

In [17]:
counts.close()
sq.close()

## When Metadata Labels Differ

`has_metadata` on a unit means an annotation file exists. The file may describe another label space. Submitters often deposit a series-level metadata table whose row labels differ from the barcode set in each matrix.

When labels differ, the join is skipped and logged. `CountMatrix.has_metadata` reports what landed, so it can be `False` on a unit whose manifest row says `True`. Turn on `INFO` logging to see the skip:

```python
import logging
logging.basicConfig(level=logging.INFO)
```